In [8]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge

In [9]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [10]:
from src.data import load_train_test

train, test = load_train_test()

In [11]:
y = np.log1p(train["SalePrice"])

X = train.drop(columns=["SalePrice", "Id"])

In [12]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [13]:
model = Ridge(alpha=10)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ]
)

In [18]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = -cross_val_score(
    pipeline,
    X,
    y,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
)
print(f"CV RMSE log mean: {scores.mean():.5f}")
print(f"CV RMSE log std:  {scores.std():.5f}")

CV RMSE log mean: 0.14679
CV RMSE log std:  0.03932


In [21]:
pipeline.fit(X, y)
X_test = test.drop(columns=["Id"])

preds_log = pipeline.predict(X_test)
preds = np.expm1(preds_log)

In [22]:
submission = pd.DataFrame(
    {
        "Id": test["Id"],
        "SalePrice": preds,
    }
)

submission.head()

,Id,SalePrice
0,1461,114305.333169
1,1462,145492.635478
2,1463,170682.737994
3,1464,192627.874861
4,1465,198253.498839


In [23]:
submission.to_csv(
    PROJECT_ROOT / "submissions" / "submission.csv",
    index=False,
)

## Итоги по базовой модели

### Модель

- Алгоритм: Ridge Regression (линейная регрессия с регуляризацией)
- Препроцессинг:
  - Числовые признаки:
    - пропуски заполнены медианой
    - выполнено масштабирование (StandardScaler)
  - Категориальные признаки:
    - пропуски заполнены самым частым значением
    - применён One-Hot Encoding

---

### Настройка обучения

- Целевая переменная:
  - использовано преобразование `log1p(SalePrice)`
- Кросс-валидация:
  - 5 фолдов
- Метрика:
  - RMSE по логарифму цены

---

### Результаты

- Средний CV RMSE (log): ~0.14
- Результат на Kaggle: **0.13375**

---

### Наблюдения

- Pipeline работает корректно и без утечек данных
- Даже простая линейная модель показывает адекватный результат
- Основные зависимости в данных (площадь, качество, гараж) уже улавливаются
- Обработка пропусков крайне примитивная (без учёта смысла признаков)

---

### Ограничения текущего решения

- Пропуски обрабатываются универсально 
- Не добавлены новые признаки
- Не удалены выбросы
- Используется только одна модель

---

### Вывод

Базовая модель даёт хороший стартовый результат и подтверждает, что:
- данные корректно подготовлены
- пайплайн построен правильно
- задача решается линейными методами